In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, desc

In [2]:
spark = SparkSession.builder.appName("StructureStreamingDemo").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/27 22:44:35 WARN Utils: Your hostname, Dhirajs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.215 instead (on interface en0)
25/08/27 22:44:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/27 22:44:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# create a object that listens to our socket 
wordsdf = spark.readStream.format("socket").option("host", "localhost").option("port", 9999).load()

25/08/27 22:46:55 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.


In [4]:
"""
line -> how was your day my day was good good is better
split the line, delimiter " "
explode -> (key, 1)
group by key, count
"""

finaldf = wordsdf.select(
    explode(
        split(wordsdf.value, " ")
    ).alias("wordsWithWeight")
)


wordCount = finaldf.groupBy("wordsWithWeight").count()

In [6]:
# output as wordCount 
# output to the console 

# output.show() # does not work, because this is a stream of data 
query = wordCount.writeStream.outputMode("complete").format("console").start()

# I want to readStream -> perform transformation -> write back my stream to the console

query.awaitTermination()

25/08/27 22:57:40 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/j6/fdnfpzrs0p900q9kmt_5nkzr0000gn/T/temporary-b9b66fbd-da75-41c1-88f1-58d9513eec6d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/08/27 22:57:40 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+---------------+-----+
|wordsWithWeight|count|
+---------------+-----+
+---------------+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+---------------+-----+
|wordsWithWeight|count|
+---------------+-----+
|            day|    2|
|            was|    2|
|            how|    1|
|           your|    1|
|             is|    1|
|             my|    1|
|         better|    1|
|           good|    2|
+---------------+-----+



-------------------------------------------
Batch: 2
-------------------------------------------
+---------------+-----+
|wordsWithWeight|count|
+---------------+-----+
|            day|    4|
|            was|    4|
|            how|    2|
|           your|    2|
|             is|    2|
|             my|    2|
|         better|    2|
|           good|    4|
+---------------+-----+



-------------------------------------------
Batch: 3
-------------------------------------------
+---------------+-----+
|wordsWithWeight|count|
+---------------+-----+
|            day|    5|
|            was|    4|
|            how|    2|
|            had|    1|
|           your|    2|
|             is|    2|
|          arman|    1|
|             my|    2|
|         better|    2|
|           good|    5|
|              a|    1|
|               |    1|
+---------------+-----+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/dhiraj/Library/Python/3.9/lib/python/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/dhiraj/Library/Python/3.9/lib/python/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socket.py", line 704, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

25/08/27 23:06:12 WARN TextSocketMicroBatchStream: Stream closed by localhost:9999


In [ ]:
# +---------------+-----+
# |wordsWithWeight|count|
# +---------------+-----+
# |            day|    4|
# |            was|    4|
# |            how|    2|
# |           your|    2|
# |             is|    2|
# |             my|    2|
# |         better|    2|
# |           good|    4|
# +---------------+-----+